In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Indic Multimodal Document Intelligence & Security Guardrails with Gemini 2.0 Flash

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/Indic_Multimodal_Document_Intelligence_with_Gemini_2.ipynb)
[![View on GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/google-gemini/cookbook/blob/main/examples/Indic_Multimodal_Document_Intelligence_with_Gemini_2.ipynb)

This recipe demonstrates how to use **Google Gemini 2.0 Flash** to extract, analyze, and explain complex technical architecture diagrams in **English**, **Tamil (தமிழ்)**, and **Hindi (हिन्दी)** with type-safe Pydantic structured output and automated security guardrails.

## Setup & Installation

In [ ]:
%pip install -U -q 'google-genai>=2.9.0' pillow pydantic

## Initialize Client & Select Model

In [ ]:
import os
try:
    from google.colab import userdata
    if "GEMINI_API_KEY" in userdata.get_keys():
        os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except (ImportError, AttributeError):
    pass

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from typing import List
from PIL import Image, ImageDraw

MODEL_ID = "gemini-2.0-flash"  # @param {type:"string"}
client = genai.Client()

## 1. Define Structured Output Schema (Pydantic)

In [ ]:
class ConceptDefinition(BaseModel):
    concept: str = Field(description="Technical concept name")
    english_definition: str = Field(description="Clear English explanation")
    tamil_definition: str = Field(description="Precise Tamil (தமிழ்) translation and breakdown")
    hindi_definition: str = Field(description="Precise Hindi (हिन्दी) translation and breakdown")

class DocumentSecurityAssessment(BaseModel):
    contains_sensitive_data: bool = Field(description="Whether the diagram exposes unencrypted credentials or PII")
    security_risk_level: str = Field(description="Risk level: Low, Medium, High, or Critical")
    security_observations: List[str] = Field(description="Key security findings or architectural notes")

class IndicDocumentAnalysisResult(BaseModel):
    document_title: str = Field(description="Title of the document or diagram")
    executive_summary_en: str = Field(description="Executive summary in English")
    executive_summary_ta: str = Field(description="Executive summary in Tamil (தமிழ்)")
    executive_summary_hi: str = Field(description="Executive summary in Hindi (हिन्दी)")
    key_concepts: List[ConceptDefinition] = Field(description="Key technical concepts in 3 languages")
    security_assessment: DocumentSecurityAssessment = Field(description="Cyber Security posture assessment")

## 2. Generate Sample Diagram & Run Multimodal Analysis

In [ ]:
# Create synthetic architecture image
img = Image.new("RGB", (800, 360), color="#0f172a")
draw = ImageDraw.Draw(img)
draw.rectangle([(0, 0), (800, 45)], fill="#1e293b")
draw.text((20, 15), "Google Cloud Zero-Trust Multi-Tier Architecture", fill="#ffffff")
draw.rounded_rectangle([(30, 70), (230, 180)], radius=8, fill="#1e3a8a", outline="#3b82f6", width=2)
draw.text((45, 85), "Client Layer\n- Web / Mobile App\n- OAuth 2.0 / JWT", fill="#ffffff")
draw.rounded_rectangle([(290, 70), (490, 180)], radius=8, fill="#064e3b", outline="#10b981", width=2)
draw.text((305, 85), "AI Gateway Layer\n- Gemini 2.0 Flash\n- Model Armor", fill="#ffffff")
draw.rounded_rectangle([(550, 70), (750, 180)], radius=8, fill="#701a75", outline="#d946ef", width=2)
draw.text((565, 85), "Backend Layer\n- Cloud Run Serverless\n- Cloud Spanner", fill="#ffffff")
draw.rounded_rectangle([(30, 210), (750, 330)], radius=8, fill="#1e293b", outline="#f59e0b", width=2)
draw.text((45, 225), "Security Policies:\n1. End-to-End mTLS encryption.\n2. Automated OSS-Fuzz vulnerability scanning.\n3. Real-time prompt injection detection.", fill="#fbbf24")
img.save("cloud_arch.png")

# Execute multimodal analysis with Gemini
prompt = "Analyze this architecture diagram. Extract bilingual summaries and security posture."
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[Image.open("cloud_arch.png"), prompt],
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=IndicDocumentAnalysisResult,
        temperature=0.2,
    ),
)

result: IndicDocumentAnalysisResult = response.parsed
print(f"Document Title: {result.document_title}")
print(f"\n[English Summary]\n{result.executive_summary_en}")
print(f"\n[Tamil Summary - தமிழ் விளக்கம்]\n{result.executive_summary_ta}")
print(f"\n[Hindi Summary - हिन्दी सारांश]\n{result.executive_summary_hi}")